# Resources misc recipes (`istari_labs_helpers`)

This notebook registers uploads as **Models** (``add_model`` / ``update_model`` via ``put_text_file``) so the backing ``File`` is a model file. **Artifacts** are a separate v2 resource type for non-model file assets; use ``.type("artifact")`` only when you intentionally listed artifacts. **Jobs**: ``platform.get_job``.

**Layout:** this notebook lives in `samples/resources/`. It looks for `samples/.env` (parent folder) the same way as the other cookbook notebooks.

**Heavy sample:** one section creates **200** models (text files) with 1–3 revisions each. Start with `BULK_COUNT = 5` to validate on your tenant before raising the count.


In [ ]:
from __future__ import annotations
from pathlib import Path

from istari_labs_helpers import IstariPlatform


def resolve_dotenv() -> str:
    cwd = Path.cwd()
    for candidate in (cwd / ".env", cwd.parent / ".env"):
        if candidate.is_file():
            return str(candidate)
    return ".env"


platform = IstariPlatform.from_env(dotenv_path=resolve_dotenv())
client = platform.client

print(platform)

## Create a **model** from an **existing** local file

`client.add_model(path, …)` registers a **Model**; the returned `Model` has `id` (model id) and `file` (backing file + revisions). Use ``platform.get_model`` or ``get_resource("model", model.id)``.

Use any small text file on disk; here we point at a sample next to the parent `samples/` folder if present.

In [ ]:
from pathlib import Path

CANDIDATES = [
    Path.cwd() / "Group3-UAS-Requirements.xlsx",
    Path.cwd().parent / "Group3-UAS-Requirements.xlsx",
]
path = next((p for p in CANDIDATES if p.is_file()), None)
if path is None:
    raise FileNotFoundError(
        "Place Group3-UAS-Requirements.xlsx under samples/ or cd to samples/ before running."
    )
print("Add model",path.name)
m = client.add_model(
    path,
    display_name=path.stem,
    external_identifier=f"cookbook-existing-model-{path.stem}",
    version_name="from-disk-v1",
)
print("model_id", m.id)
print("file_id", m.file.id)
print("latest revision", m.file.revisions[-1].id, m.file.revisions[-1].version_name)

## `put_text_file` — text in memory → new **model** or new revision

`IstariPlatform.put_text_file` writes UTF-8 text to a temp file, then calls `add_model` (**new model**) or `update_model` (**new revision** when you pass `model_id=…`).

Optional kwargs mirror the client: `display_name`, `external_identifier`, `version_name`, `description`. If `filename` has no suffix, `.txt` is appended.

In [ ]:
first = platform.put_text_file(
    "hello from put_text_file\n",
    filename="cookbook-put-text-demo.txt",
    display_name="Cookbook put_text demo",
    external_identifier="cookbook-put-text-ext",
    version_name="v-one",
)
second = platform.put_text_file(
    "second revision line\n",
    filename="cookbook-put-text-demo.txt",
    model_id=first.id,
    version_name="v-two",
)
print("same model", first.id == second.id)
print("revisions", [r.version_name for r in second.file.revisions])

## Bulk create **N** models (1–3 revisions each)

Each row is a **Model** (via ``put_text_file`` → ``add_model`` / ``update_model``) with a unique basename and `external_identifier`. File **content** includes the basename and revision label for verification.

Set **`BULK_COUNT`** to a small number first; **`200`** is the default target when you are ready.

In [ ]:
import time

BULK_COUNT = 200
BULK_TAG = "labs-helpers-bulk-demo"

bulk_meta: list[tuple[str, str, str, str, list[str]]] = []
for i in range(BULK_COUNT):
    base = f"{BULK_TAG}-{i:04d}"
    ext_id = f"{BULK_TAG}-ext-{i:04d}"
    n_rev = (i % 3) + 1  # cycles 1..3 so row 0001 always has a v2, row 0002 a v3
    names: list[str] = []
    body = (
        f"bulk_resource={base}\n"
        f"bulk_external_id={ext_id}\n"
        f"bulk_revision=1\n"
    )
    f = platform.put_text_file(
        body,
        filename=f"{base}.txt",
        display_name=base,
        external_identifier=ext_id,
        version_name="v1",
    )
    names.append("v1")
    for r in range(2, n_rev + 1):
        body_r = (
            f"bulk_resource={base}\n"
            f"bulk_external_id={ext_id}\n"
            f"bulk_revision={r}\n"
        )
        f = platform.put_text_file(
            body_r,
            filename=f"{base}.txt",
            model_id=f.id,
            version_name=f"v{r}",
        )
        names.append(f"v{r}")
    bulk_meta.append((base, ext_id, "model", f.id, names))
    if i and i % 25 == 0:
        time.sleep(0.4)

print("models created:", len(bulk_meta))
print("example:", bulk_meta[0])

## Search resources **by file name**

`list_resources` accepts **list-valued** filters (`file_name=[...]`). This notebook’s uploads are **models** — use **``.type("model")``**.


In [ ]:
fname = f"{BULK_TAG}-0000.txt"
q = platform.resources().type("model").filter(file_name=[fname])
hits = list(iter(q))
print("count", len(hits))
for hit in hits[:5]:
    print(hit.id, hit.name, hit.extension, hit.external_identifier, hit.version_name)

## Search **by external identifier**

Use the same pattern with `external_identifier=["your-key"]`. External ids are stored on the **current** revision metadata surfaced in search rows.

In [ ]:
ext = f"{BULK_TAG}-ext-0000"
q = platform.resources().type("model").filter(external_identifier=[ext])
for hit in list(iter(q))[:5]:
    print(hit.id, hit.file_revision_id, hit.version_name, hit.external_identifier)

## Fetch a **revision** by id (and check the parent file / model)

- `platform.get_revision(revision_id)` → `FileRevision` (content tokens, `file_id`, …).
- For a **Model**, `model.file.id` should equal `revision.file_id` when the revision belongs to that model’s backing file.
- `client.get_file_by_revision_id` exists if you need the `File` envelope directly.

Below we take `file_revision_id` from a `list_resources` hit on a **model**, fetch the revision, and compare to `get_file` using `revision.file_id`.

In [ ]:
hit = next(iter(platform.resources().type("model").filter(file_name=[fname])), None)
if hit is None:
    raise RuntimeError("Run the bulk section (or adjust fname) first.")

rev = platform.get_revision(hit.file_revision_id)
fetched_file = client.get_file(rev.file_id)
print("revision", rev.id, "file_id", rev.file_id)
print("matches file.id", fetched_file.id == rev.file_id)

# Optional: wrap the model and read text via ModelView / ResourceView
view = platform.get_resource(hit.type_name, hit.id)
print("model view", view)
print(view.read_text()[:200])

## Model-backed file: revision id + **model id**

Models also expose a `file` with revisions. After `add_model`, latest revision id should round-trip through `get_revision` and share the same `file_id` as `model.file.id`.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as td:
    p = Path(td) / "model-rev-check.txt"
    p.write_text("model revision linkage check\n", encoding="utf-8")
    model = client.add_model(
        path=p,
        external_identifier="cookbook-model-rev-link",
        display_name="Cookbook model rev link",
        version_name="m-v1",
    )

rev_id = model.file.revisions[-1].id
rev = platform.get_revision(rev_id)
print("model_id", model.id, "file_id", model.file.id)
print("revision.file_id == model.file.id", rev.file_id == model.file.id)
mv = platform.get_model(model.id)
print(mv.read_text())

## Search by **external identifier** + **version name**

Combine filters in one `filter(...)` call. Values are lists per the OpenAPI schema.

In [ ]:
q = platform.resources().type("model").filter(
    external_identifier=[f"{BULK_TAG}-ext-0001"],
    version_name=["v2"],
)
for hit in list(iter(q))[:10]:
    print(hit.file_revision_id, hit.version_name, hit.external_identifier)

## Search by **file name** + **version name**

Useful when external id is shared across revisions but version labels differ.

In [ ]:
q = platform.resources().type("model").filter(
    file_name=[f"{BULK_TAG}-0002.txt"],
    version_name=["v3"],
)
rows = list(iter(q))
print("hits", len(rows))
for hit in rows[:10]:
    print(hit)